# Installations and setup

In [1]:
# https://stackoverflow.com/questions/51342408/how-do-i-install-python-packages-in-googles-colab

# https://discuss.huggingface.co/t/cannot-user-load-dataset-in-google-colab/83525/5

# https://stackoverflow.com/questions/51342408/how-do-i-install-python-packages-in-googles-colab

# https://colab.research.google.com/github/google-health/medgemma/blob/main/notebooks/quick_start_with_hugging_face.ipynb (for using
# huggingface in Google Colab)


%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

%pip install -U transformers

%pip install datasets

%pip install accelerate

%pip install nltk

%pip install python-Levenshtein

%pip install bert-score

%pip install tiktoken

%pip install protobuf

%pip install torchmetrics[text]

%pip install hdm2 --quiet

%pip install -U sentence-transformers

%pip install setfit

# https://discuss.huggingface.co/t/cannot-user-load-dataset-in-google-colab/83525/4
%pip install -U huggingface_hub

%pip install numpy==2.2.6
# https://github.com/huggingface/xet-core/issues/483
%pip install -U hf_xet

Looking in indexes: https://download.pytorch.org/whl/cu128


In [2]:
import os
import sys
from google.colab import userdata

google_colab = "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT")

# Use secret if running in Google Colab
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Print Versions of Libaries

In [3]:
import numpy
import pandas
import matplotlib
import nltk
import scipy
import sklearn
import statsmodels
import transformers
import datasets
import torch
import torchmetrics
import sentence_transformers
import huggingface_hub


print("Numpy Version = {}".format(numpy.__version__))
print("Pandas Version = {}".format(pandas.__version__))
print("Matplotlib Version = {}".format(matplotlib.__version__))
print("NLTK Version = {}".format(nltk.__version__))
print("Scipy Version = {}".format(scipy.__version__))
print("Sklearn Version = {}".format(sklearn.__version__))
print("Statsmodels Version = {}".format(statsmodels.__version__))
print("Transformers Version = {}".format(transformers.__version__))
print("Datasets Version = {}".format(datasets.__version__))
print("Torch Version = {}".format(torch.__version__))
print("TorchMetrics Version = {}".format(torchmetrics.__version__))
print("Sentence Transformers Version = {}".format(sentence_transformers.__version__))
print("Huggingface Hub Version = {}".format(huggingface_hub.__version__))

Numpy Version = 2.2.6
Pandas Version = 2.2.2
Matplotlib Version = 3.10.0
NLTK Version = 3.9.1
Scipy Version = 1.16.3
Sklearn Version = 1.6.1
Statsmodels Version = 0.14.6
Transformers Version = 5.5.0
Datasets Version = 4.0.0
Torch Version = 2.10.0+cu128
TorchMetrics Version = 1.9.0
Sentence Transformers Version = 5.3.0
Huggingface Hub Version = 1.9.2


# Imports

In [4]:
from FileHandler import FileHandler
from Evaluater import Evaluater
from ZeroShotPrompter import ZeroShotPrompter
from FewShotPrompter import FewShotPrompter
from listutils import list_distinct
import pandas as pd
from tqdm import tqdm
import hyperparameters
import nltk

nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

# Unzipping Data

In [5]:
# https://www.geeksforgeeks.org/machine-learning/how-to-upload-folders-to-google-colab/
!unzip Storage.zip

Archive:  Storage.zip
   creating: Storage - Correct/CorrelationAnalysis/
   creating: Storage - Correct/CorrelationAnalysis/Squad_Dummy/
  inflating: Storage - Correct/CorrelationAnalysis/Squad_Dummy/cache_test.csv  
  inflating: Storage - Correct/CorrelationAnalysis/Squad_Dummy/cache_train.csv  
   creating: Storage - Correct/CorrelationAnalysis/Squad_Dummy/Feature_to_Feature/
  inflating: Storage - Correct/CorrelationAnalysis/Squad_Dummy/Feature_to_Feature/Answer_Start_Relative_to_Perplexity.png  
  inflating: Storage - Correct/CorrelationAnalysis/Squad_Dummy/Feature_to_Feature/Answer_Start_Relative_to_question_type.png  
  inflating: Storage - Correct/CorrelationAnalysis/Squad_Dummy/Feature_to_Feature/Answer_Start_Relative_to_y_question_BERT.png  
  inflating: Storage - Correct/CorrelationAnalysis/Squad_Dummy/Feature_to_Feature/Answer_Start_Relative_to_y_question_EditDist.png  
  inflating: Storage - Correct/CorrelationAnalysis/Squad_Dummy/Feature_to_Feature/Answer_Start_Relative_t

# Flan T5 inline, since for some reason does not work with device when in class:

In [6]:

import torch

# https://huggingface.co/google/flan-t5-xxl/discussions/41
# https://github.com/huggingface/transformers/issues/5204
# https://community.deeplearning.ai/t/flant5-maximum-input-and-output-length/421273
# https://huggingface.co/docs/transformers/main/en/model_doc/t5#transformers.T5Config
# https://discuss.huggingface.co/t/passing-inputs-longer-than-512-tokens-after-pretraining-a-t5-model-is-it-safe/170655/3

class FlanT5:

    def __init__(self, model, tokenizer):
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        print("Device = {}".format(self.device))

        self.model = model
        self.tokenizer = tokenizer

        self.tokenizer.model_max_length = 4000
        self.model.generation_config.do_sample = False
        self.PAD_ID = self.tokenizer.pad_token_id
        assert(self.PAD_ID == 0)


    # https://huggingface.co/docs/transformers/main_classes/text_generation
    # https://discuss.huggingface.co/t/how-can-i-obtain-the-logits-via-model-generate/110636/2
    # https://www.kaggle.com/discussions/questions-and-answers/557651
    # https://discuss.pytorch.org/t/convert-a-tuple-into-tensor/82964/2
    # https://huggingface.co/google/gemma-2b-it/discussions/55
    # https://www.kaggle.com/discussions/questions-and-answers/557651
    def answer(self, prompts):
        # Returns input_ids and attention_mask. Pads to the longest prompt
        inputs = self.tokenizer(prompts, return_tensors = "pt", padding = "longest")
        inputs = inputs.to(self.device)

        num_new_tokens = 40
        # https://github.com/huggingface/transformers/issues/12503
        # https://discuss.huggingface.co/t/output-dimension-of-automodelforcausallm/47225
        # https://huggingface.co/docs/transformers/internal/generation_utils
        # https://huggingface.co/TheBloke/Llama-2-70B-Chat-GPTQ/discussions/25
        outputs = self.model.generate(**inputs, max_new_tokens = num_new_tokens, output_logits = True, return_dict_in_generate = True, do_sample = False)
        # Output.sequences has shape [batch_size, num_input_tokens + num_generated_tokens]
        generated_tokens = outputs.sequences

        answer_lengths = []
        B = generated_tokens.shape[0]
        for i in range(B):
            tokens = generated_tokens[i, :]
            tokenids_stripped = [token_id for token_id in tokens if token_id != self.PAD_ID]
            answer_lengths.append(len(tokenids_stripped))


        answers = self.tokenizer.batch_decode(generated_tokens, skip_special_tokens = True)
        #answers = [answer.strip() for answer in answers]
        # Converts tuple of (batch_size, vocab_size) to (num_generated_tokens, batch_size, vocab_size)
        logits = self.tuple_of_tensors_to_tensor(outputs.logits)

        #print("Generated Tokens Shape = {}".format(generated_tokens.shape))
        #print("Logits shape = {}".format(logits.shape))

        return answers, logits.cpu().detach(), answer_lengths


    # https://discuss.huggingface.co/t/how-to-get-cross-attention-values-of-t5/970/3
    # https://discuss.huggingface.co/t/what-the-tokens-are-cross-attentions-output-for/14420
    # https://huggingface.co/docs/transformers/v4.26.1/model_doc/t5
    # https://discuss.huggingface.co/t/t5-why-do-we-have-more-tokens-expressed-via-cross-attentions-than-the-decoded-sequence/31893
    def cross_attention_map(self, prompts):
        # Returns input_ids and attention_mask. Pads to the longest prompt
        num_new_tokens = 40
        # https://github.com/huggingface/transformers/issues/12503
        # https://discuss.huggingface.co/t/output-dimension-of-automodelforcausallm/47225
        # https://huggingface.co/docs/transformers/internal/generation_utils
        # https://huggingface.co/TheBloke/Llama-2-70B-Chat-GPTQ/discussions/25


        answers = []
        output_attentions = []
        inputs_tokens = []

        for prompt in prompts:
            input = self.tokenizer(prompt, return_tensors = "pt")
            input_tokens = self.tokenizer.tokenize(prompt, return_tensors = "pt")
            inputs_tokens.append(input_tokens)

            input = input.to(self.device)
            output = self.model.generate(**input, max_new_tokens = num_new_tokens, output_attentions = True, return_dict_in_generate = True, do_sample = False)
            generated_tokens = output.sequences
            cross_attention_map = output.cross_attentions

            answer = self.tokenizer.batch_decode(generated_tokens, skip_special_tokens = True)[0]


            answers.append(answer)
            output_attentions.append(cross_attention_map)

        return answers, output_attentions, inputs_tokens

    '''
        sentence: A string

        Tokenizes the sentence and returns the result
    '''
    def tokenize(self, sentence):
        return self.tokenizer(sentence, return_tensors = "pt")


    # https://discuss.pytorch.org/t/convert-a-tuple-into-tensor/82964/2
    def tuple_of_tensors_to_tensor(self, tuple_of_tensors):
        return torch.stack(list(tuple_of_tensors), dim=0)

# Setup

In [7]:
'''
    dataset: A dataset of the custom dataset class (must implement getName).
    qids: List of query ids.
'''
def get_retrieved_contexts(data_iden, qids):
    file_handler = FileHandler()

    data_set_name = data_iden.split("_")[0]

    if data_set_name == "Squad":
        (b, k1) = hyperparameters.BM25_default()
    elif data_set_name == "WikiMultiHop":
        (b, k1) = hyperparameters.BM25_default()
    else:
        raise Exception("Unknown Dataset name. Must be either Squad or WikiMultiHop")

    # A df with columns qid, docno, score, rank
    retrieval_df = file_handler.load_df(dir = "RetrievalFiles", dataset_name = data_iden, filename = "ModelFiles/RetrievalResults_b={}_k1={}.csv".format(str(b), str(k1)))
    # A df with columns docno, text
    docs_df = file_handler.load_df(dir = "DataFiles", dataset_name = data_iden, filename = "docs.csv")
    qrels_df = file_handler.load_df(dir = "DataFiles", dataset_name = data_iden, filename = "qrels.csv")

    assert(list(qrels_df["qid"]) == qids)

    # Construct mapping of docno to context
    docno_to_context = {}

    for (docno, context) in zip(list(docs_df["docno"]), list(docs_df["text"])):
        docno_to_context[docno] = context

    # ASSUMPTION: ASSUME K = 1
    qid_to_docno = {}

    retrieval_qids = list(retrieval_df["qid"])
    retrieval_docnos = list(retrieval_df["docno"])

    assert(len(retrieval_qids) == len(retrieval_docnos))

    # https://stackoverflow.com/questions/522372/finding-first-and-last-index-of-some-value-in-a-list-in-python
    print("Finding RAG docs ...")
    for qid in tqdm(list_distinct(retrieval_qids)):
        # Find first index of qids in retrieval_qids and use it find the first relevant document
        first_index = retrieval_qids.index(qid)
        # Find the docno that corresponds to the line
        qid_to_docno[qid] = retrieval_docnos[first_index]

    contexts = []
    RAG_docnos = []
    true_docnos = list(qrels_df["docno"])

    retrieved_qids = list_distinct(retrieval_qids)

    for qid in qids:
        if qid in retrieved_qids:
            docno = qid_to_docno[qid]
            retrieved_context = str(docno_to_context[docno])
            RAG_docnos.append(docno)
            contexts.append(retrieved_context)
        # For qids, which are not listed in the retrieval file.
        else:
            contexts.append("")
            RAG_docnos.append(-1)


    return list(contexts), list(RAG_docnos), true_docnos

In [8]:
'''
    TODO: Modify to also evaluate the RAG model.
'''
def runEval(data_iden, prompt_engineer, model, qids, questions, true_contexts, true_answers):
    RAG_contexts, RAG_docnos, true_docnos = get_retrieved_contexts(data_iden, qids)

    RAG_qids = []
    RAG_delete_qids = []

    for i,(RAG_docno, true_docno) in enumerate(zip(RAG_docnos, true_docnos)):
        if int(RAG_docno) != int(true_docno) and int(RAG_docno) != -1:
            RAG_qids.append(qids[i])
        elif int(RAG_docno) == -1:
            RAG_delete_qids.append(qids[i])

    print("Number of faulty retrieved documents is {}".format(len(RAG_qids)))
    print("Number of Empty retrievals is {}".format(len(RAG_delete_qids)))
    print("Qids of Faulty Retrieved is {}".format(RAG_delete_qids))

    qids_RAG = [qid for qid in RAG_qids]
    questions_RAG = [question for i,question in enumerate(questions) if i in RAG_qids]
    contexts_RAG = [context for i,context in enumerate(RAG_contexts) if i in RAG_qids]
    true_answers_RAG = [answer for i,answer in enumerate(true_answers) if i in RAG_qids]

    prompts_M_question = prompt_engineer.construct_prompts_without_context(questions)
    prompts_M_RAG = prompt_engineer.construct_prompts_with_context(questions_RAG, contexts_RAG)
    prompts_M_context = prompt_engineer.construct_prompts_with_context(questions, true_contexts)

    assert(len(prompts_M_question) == len(prompts_M_context))
    assert(len(prompts_M_context) != len(prompts_M_RAG))

    evaluater = Evaluater()
    file_handler = FileHandler()

    # https://saturncloud.io/blog/how-to-write-a-pandas-dataframe-to-a-txt-file/
    results_M_question = evaluater.evaluate(model, prompts_M_question, true_answers, qids)
    file_handler.save_df(df = results_M_question, dir = "EvaluationFiles",  dataset_name = data_iden ,filename = "M_question.csv", header = True)

    results_M_context = evaluater.evaluate(model, prompts_M_context, true_answers, qids, batch_size = 1)
    file_handler.save_df(df = results_M_context, dir = "EvaluationFiles", dataset_name = data_iden , filename = "M_context.csv", header = True)

    assert(len(prompts_M_RAG) == len(true_answers_RAG))
    assert(len(true_answers_RAG) == len(qids_RAG))
    results_M_RAG = evaluater.evaluate(model, prompts_M_RAG, true_answers_RAG, qids_RAG, batch_size = 1)
    # https://stackoverflow.com/questions/53727153/replacing-rows-in-pandas-dataframe-with-other-dataframe-based-on-index
    # https://stackoverflow.com/questions/39267372/replace-rows-in-a-pandas-df-with-rows-from-another-df
    # Replace rows in df_context, with rows of M_RAG, where the fetched document is wrong.
    #cols = list(results_M_context.columns)
    #results_M_context.loc[results_M_context.qid.isin(results_M_RAG.qid), cols] = results_M_RAG[cols]
    # https://stackoverflow.com/questions/22430195/pandas-dataframe-set-rows-equal-to-other-rows

    results_M_context[results_M_context["qid"].isin(qids_RAG)] = results_M_RAG.values
    # https://medium.com/@amit25173/filtering-data-in-pandas-basics-you-need-to-know-639ed999821b
    results_M_context = results_M_context[~results_M_context["qid"].isin(RAG_delete_qids)]

    file_handler.save_df(df = results_M_context, dir = "EvaluationFiles",  dataset_name = data_iden ,filename = "M_RAG.csv", header = True)

# Running the Experiment

In [9]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-xl", device_map = "cuda")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-xl", device_map = "cuda")

flan = FlanT5(model, tokenizer)

prompter = ZeroShotPrompter()
data_set_iden = "Squad_Tune"

file_handler = FileHandler()

df = file_handler.load_df(dir = "DataFiles", dataset_name = data_set_iden, filename = "collab.csv")

qids = list(df["qid"])
questions = list(df["question"])
contexts = list(df["context"])
answers = list(df["answer"])

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device = cuda:0
/content/Storage - Correct


# Remember:

### 1): Set Dataset

### 2): Set BM25 parameters

### 3): Set Prompter

In [10]:
runEval(data_set_iden, prompter, flan, qids, questions, contexts, answers)

/content/Storage - Correct
Finding RAG docs ...


100%|██████████| 2500/2500 [00:00<00:00, 6622.83it/s] 


Number of faulty retrieved documents is 372
Number of Empty retrievals is 0
Qids of Faulty Retrieved is []


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Time total to run evaluate = 10.24916579325994
Total Inference time = 9.722547396024067


100%|██████████| 2500/2500 [23:02<00:00,  1.81it/s]


Time total to run evaluate = 23.044167149066926
Total Inference time = 22.536592050393423


100%|██████████| 372/372 [03:03<00:00,  2.03it/s]

Time total to run evaluate = 3.0522892673810325
Total Inference time = 2.972147297859192
